In [1]:
# --- Setup Environment for ArcFace ---
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu insightface chromadb scikit-learn psutil pandas matplotlib tqdm opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61

In [1]:
import os
import sys
import time
import re
import subprocess
import warnings
import psutil
import logging
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import cv2
import chromadb
from google.colab import drive

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# OFFICIAL INSIGHTFACE IMPLEMENTATION (SCRFD + ARCFACE)
import insightface
from insightface.app import FaceAnalysis

warnings.filterwarnings("ignore")

In [2]:
drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [3]:
# --- Logging Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [%(levelname)s] - %(message)s'
)
logger = logging.getLogger(__name__)

In [4]:
# --- Paths & Constants ---
PROJECT_PATH = "/content/drive/MyDrive/face_recognition"
ENROLLMENT_PATH = os.path.join(PROJECT_PATH, "dataset", "enrollment")
TESTING_PATH = os.path.join(PROJECT_PATH, "dataset", "testing")
DATABASE_PATH = os.path.join(PROJECT_PATH, "chroma_db", "arcface_db")
COLLECTION_NAME = "arcface_faces"
MODEL_NAME = "ArcFace"
TOP_K = 1

BENCHMARK_DIR = os.path.join(PROJECT_PATH, "benchmark_results", "arcface")
os.makedirs(BENCHMARK_DIR, exist_ok=True)

# Set to True to wipe and re-index ChromaDB on every run
FORCE_REENROLL = True

In [5]:
# --- Initialize Hardware Monitoring & InsightFace ---
logger.info("Initializing Official ArcFace (InsightFace SCRFD + ArcFace) Pipeline...")
start_model_load = time.perf_counter()

# Initialize InsightFace FaceAnalysis using CUDA if available
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if torch.cuda.is_available() else ['CPUExecutionProvider']
app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=0, det_size=(640, 640))

MODEL_LOADING_TIME = time.perf_counter() - start_model_load

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:07<00:00, 38631.90KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [6]:
# --- Helper Functions ---
def read_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")
    return image

def get_person_name(image_name):
    base = os.path.splitext(image_name)[0].lower()
    return re.sub(r'\d+$', '', base).strip('_')

def get_expected_identities(image_name):
    base = os.path.splitext(image_name)[0].lower()

    # Custom multi-face file handling if applicable
    if "dr_strange" in base and "loki" in base:
        return ["dr_strange", "loki"]
    if "tony" in base and "natasha" in base:
        return ["tony", "natasha"]
    if "tony" in base and "steve" in base:
        return ["tony", "steve_roger"]

    name = re.sub(r'\d+$', '', base).strip('_')
    if "unknown" in name:
        return ["Unknown"]
    return [name]

def get_cpu_usage():
    return psutil.cpu_percent(interval=None)

def get_ram_usage():
    return psutil.virtual_memory().percent

def get_gpu_metrics():
    util = 0.0
    vram_mb = 0.0
    try:
        util_raw = subprocess.getoutput("nvidia-smi --query-gpu=utilization.gpu --format=csv,nounits,noheader")
        vram_raw = subprocess.getoutput("nvidia-smi --query-gpu=memory.used --format=csv,nounits,noheader")

        util_match = re.findall(r"\d+", util_raw)
        vram_match = re.findall(r"\d+", vram_raw)

        util = float(util_match[-1]) if util_match else 0.0
        vram_mb = float(vram_match[-1]) if vram_match else 0.0
    except (ValueError, IndexError):
        pass

    return util, vram_mb

def get_model_size_mb():
    total_size_bytes = 0
    model_root = os.path.expanduser('~/.insightface/models/buffalo_l')
    if os.path.exists(model_root):
        for path, dirs, files in os.walk(model_root):
            for f in files:
                fp = os.path.join(path, f)
                total_size_bytes += os.path.getsize(fp)
    return total_size_bytes / (1024 * 1024)

logger.info(f"Model Loaded in: {MODEL_LOADING_TIME:.4f} sec (Size: {get_model_size_mb():.2f} MB)")

In [7]:
# Initialize Vector DB
client = chromadb.PersistentClient(path=DATABASE_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"} # Official Cosine metric space
)

In [8]:
# --- Enrollment Pipeline ---
def enroll_images():
    logger.info("--- Starting ArcFace Enrollment ---")
    image_files = sorted(os.listdir(ENROLLMENT_PATH))

    for image_name in tqdm(image_files, desc="Enrolling"):
        image_path = os.path.join(ENROLLMENT_PATH, image_name)
        try:
            image = read_image(image_path)
        except ValueError:
            continue

        # SCRFD detection + SCRFD 5-point alignment + ArcFace embedding extraction
        faces = app.get(image)
        if len(faces) != 1:
            logger.warning(f"Skipped {image_name}: Found {len(faces)} faces (enrollment requires exactly 1 face).")
            continue

        # InsightFace automatically returns L2-normalized 512-D float32 embedding
        embedding = faces[0].embedding.tolist()
        person_name = get_person_name(image_name)
        bbox = faces[0].bbox.astype(int).tolist()

        collection.upsert(
            ids=[image_name],
            embeddings=[embedding],
            metadatas=[{
                "person_name": person_name,
                "image_name": image_name,
                "image_path": image_path,
                "bbox": str(bbox),
                "model": MODEL_NAME
            }]
        )

    logger.info(f"Enrollment Complete! Total enrolled identities: {collection.count()}")

In [11]:
# --- Query & Search Pipeline ---
def search_image(image_path):
    try:
        image = read_image(image_path)
    except ValueError as e:
        logger.warning(f"Cannot read image {image_path}: {e}")
        return []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0_feat = time.perf_counter()

    # Detector (SCRFD) + Landmark Alignment + ArcFace forward pass
    faces = app.get(image)
    if len(faces) == 0:
        return []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    feature_extraction_time = time.perf_counter() - t0_feat

    predictions = []
    total_search_time = 0.0

    for face in faces:
        query_embedding = face.embedding.tolist()

        t0_search = time.perf_counter()
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=TOP_K,
            include=["metadatas", "distances"]
        )
        search_time = time.perf_counter() - t0_search
        total_search_time += search_time

        if not results["distances"] or not results["distances"][0]:
            best_match = "Unknown"
            distance = float('inf')
        else:
            distance = results["distances"][0][0] # Cosine Distance (1 - Cosine Similarity)
            best_match = results["metadatas"][0][0]["person_name"].replace(" ", "_")

        predictions.append({
            "Best_Match": best_match,
            "Cosine_Distance": distance,
            "Feature_Extraction_Time": feature_extraction_time,
            "Search_Time": search_time,
        })

    total_inference = feature_extraction_time + total_search_time
    for p in predictions:
        p["Inference_Time"] = total_inference

    return predictions

In [12]:
def evaluate_system():
    if collection.count() == 0:
        logger.error("Database is empty. Evaluation aborted.")
        return

    logger.info("--- Starting ArcFace Testing & Performance Evaluation ---")

    test_files = sorted(os.listdir(TESTING_PATH))

    all_searches = []
    cpu_records = []
    ram_records = []
    gpu_util_records = []
    global_peak_vram = 0.0

    get_cpu_usage() # Warmup CPU reading

    # 1. Execute raw inference across testing dataset
    for image_name in test_files:
        image_path = os.path.join(TESTING_PATH, image_name)
        expected_identities = [name.replace(" ", "_") for name in get_expected_identities(image_name)]
        preds = search_image(image_path)

        all_searches.append({
            "image_name": image_name,
            "preds": preds,
            "expected": expected_identities
        })

        cpu_records.append(get_cpu_usage())
        ram_records.append(get_ram_usage())
        g_util, g_vram = get_gpu_metrics()
        gpu_util_records.append(g_util)

        if g_vram > global_peak_vram:
            global_peak_vram = g_vram

    # 2. Automated 1D Threshold Search (Maximizing Weighted F1-Score)
    COSINE_THRESHOLDS = np.arange(0.15, 0.85, 0.025)

    best_f1 = -1.0
    best_cos_threshold = 1.0
    best_raw_predictions = []
    best_y_true = []
    best_y_pred = []

    logger.info("Running automated threshold optimization for ArcFace Cosine Distance...")

    for cos_thresh in COSINE_THRESHOLDS:
        temp_raw = []
        temp_y_true = []
        temp_y_pred = []

        for item in all_searches:
            expected = item["expected"].copy()
            preds = item["preds"]
            frame_preds = []

            for p in preds:
                p_copy = p.copy()
                p_copy["Image_Name"] = item["image_name"]

                is_match = p_copy["Best_Match"] in expected
                is_close = p_copy["Cosine_Distance"] <= cos_thresh

                if is_match and is_close:
                    p_copy["Ground_Truth"] = p_copy["Best_Match"]
                    expected.remove(p_copy["Best_Match"])
                else:
                    p_copy["Ground_Truth"] = None
                frame_preds.append(p_copy)

            for p_copy in frame_preds:
                if p_copy["Ground_Truth"] is None:
                    if expected:
                        p_copy["Ground_Truth"] = expected.pop(0)
                    else:
                        p_copy["Ground_Truth"] = "Spurious Detection"

            temp_raw.extend(frame_preds)

            for missing in expected:
                temp_raw.append({
                    "Image_Name": item["image_name"],
                    "Ground_Truth": missing,
                    "Best_Match": "Missed",
                    "Cosine_Distance": float('inf'),
                    "Feature_Extraction_Time": 0.0,
                    "Search_Time": 0.0,
                    "Inference_Time": 0.0
                })

        for p in temp_raw:
            if p["Best_Match"] == "Missed":
                temp_y_pred.append("Unknown (Missed)")
                temp_y_true.append(p["Ground_Truth"])
            else:
                final_pred = p["Best_Match"] if p["Cosine_Distance"] <= cos_thresh else "Unknown"
                temp_y_pred.append(final_pred)
                temp_y_true.append(p["Ground_Truth"])

        current_f1 = f1_score(temp_y_true, temp_y_pred, average="weighted", zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_cos_threshold = cos_thresh
            best_raw_predictions = temp_raw
            best_y_true = temp_y_true
            best_y_pred = temp_y_pred

    logger.info(f"Optimal Threshold Selected -> Cosine Distance: {best_cos_threshold:.3f} (F1: {best_f1*100:.2f}%)")

    # 3. Compile predictions and metrics for CSV export
    final_csv_data = []

    for item in best_raw_predictions:
        if item["Best_Match"] == "Missed":
            final_pred = "Unknown (Missed)"
            status = "Failed"
            correct_flag = "Incorrect"
        else:
            final_pred = item["Best_Match"] if item["Cosine_Distance"] <= best_cos_threshold else "Unknown"
            status = "Known" if final_pred != "Unknown" else "Unknown"
            correct_flag = "Correct" if final_pred == item["Ground_Truth"] else "Incorrect"

        final_csv_data.append({
            "Image_Name": item["Image_Name"],
            "Ground_Truth": item["Ground_Truth"],
            "Prediction": final_pred,
            "Status": status,
            "Cosine_Distance": round(item["Cosine_Distance"], 4) if item["Cosine_Distance"] != float('inf') else "N/A",
            "Correct/Incorrect": correct_flag,
            "Feature_Extraction_Time": item["Feature_Extraction_Time"],
            "Search_Time": item["Search_Time"],
            "Inference_Time": item["Inference_Time"]
        })

    df_preds = pd.DataFrame(final_csv_data)
    df_preds.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_predictions.csv"), index=False)

    accuracy = accuracy_score(best_y_true, best_y_pred)
    precision = precision_score(best_y_true, best_y_pred, average="weighted", zero_division=0)
    recall = recall_score(best_y_true, best_y_pred, average="weighted", zero_division=0)

    labels = sorted(list(set(best_y_true) | set(best_y_pred)))
    cm = confusion_matrix(best_y_true, best_y_pred, labels=labels)
    df_cm = pd.DataFrame(cm, index=labels, columns=labels)
    df_cm.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_confusion_matrix.csv"))

    avg_feat_time = df_preds[df_preds["Feature_Extraction_Time"] > 0]["Feature_Extraction_Time"].mean()
    avg_search_time = df_preds[df_preds["Search_Time"] > 0]["Search_Time"].mean()
    avg_inference_time = df_preds[df_preds["Inference_Time"] > 0]["Inference_Time"].mean()

    avg_cpu = sum(cpu_records) / len(cpu_records) if cpu_records else 0.0
    avg_ram = sum(ram_records) / len(ram_records) if ram_records else 0.0
    avg_gpu_util = sum(gpu_util_records) / len(gpu_util_records) if gpu_util_records else 0.0

    if global_peak_vram > 0 or avg_gpu_util > 0:
        gpu_usage_str = f"Util: {avg_gpu_util:.1f}%, Peak VRAM: {global_peak_vram:.0f} MB"
    else:
        gpu_usage_str = "No GPU / CUDA Idle"

    results_data = [{
        "Model": MODEL_NAME,
        "Selected_Cosine_Distance_Threshold": round(best_cos_threshold, 3),
        "Accuracy_Percent": round(accuracy * 100, 2),
        "Precision_Percent": round(precision * 100, 2),
        "Recall_Percent": round(recall * 100, 2),
        "F1_Score_Percent": round(best_f1 * 100, 2),
        "Average_Feature_Extraction_Time_sec": round(avg_feat_time if not pd.isna(avg_feat_time) else 0, 6),
        "Average_Search_Time_sec": round(avg_search_time if not pd.isna(avg_search_time) else 0, 6),
        "Average_Inference_Time_sec": round(avg_inference_time if not pd.isna(avg_inference_time) else 0, 6),
        "Model_Loading_Time_sec": round(MODEL_LOADING_TIME, 6),
        "Embedding_Dimension": 512,
        "Model_Size_MB": round(get_model_size_mb(), 2),
        "CPU_Usage_Percent": round(avg_cpu, 2),
        "RAM_Usage_Percent": round(avg_ram, 2),
        "GPU_VRAM_Usage": gpu_usage_str
    }]

    df_results = pd.DataFrame(results_data)
    df_results.to_csv(os.path.join(BENCHMARK_DIR, f"{MODEL_NAME.lower()}_results.csv"), index=False)

    logger.info("\n" + "="*60)
    logger.info("         ARCFACE BENCHMARK PERFORMANCE REPORT")
    logger.info("="*60)
    logger.info(f"Total Faces Evaluated : {len(best_y_true)}")
    logger.info(f"Optimum Cosine Thresh : {best_cos_threshold:.3f}")
    logger.info(f"System Accuracy       : {accuracy * 100:.2f}%")
    logger.info(f"System Precision      : {precision * 100:.2f}%")
    logger.info(f"System Recall         : {recall * 100:.2f}%")
    logger.info(f"System F1-Score       : {best_f1 * 100:.2f}%")
    logger.info("-" * 60)
    logger.info(f"Avg Inference Time    : {round(avg_inference_time if not pd.isna(avg_inference_time) else 0, 4)} sec")
    logger.info(f"GPU Monitor Status    : {gpu_usage_str}")
    logger.info("="*60)
    logger.info(f"Data Exported to: {BENCHMARK_DIR}")

# --- Main Driver ---
if __name__ == "__main__":
    if FORCE_REENROLL:
        # Delete the collection directly from the client and recreate it
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass

        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )
        enroll_images()
    elif collection.count() == 0:
        enroll_images()

    evaluate_system()

Enrolling: 100%|██████████| 7/7 [00:17<00:00,  2.55s/it]
